# SPECIL Model Pipeline

Build SPECIL train/test data, train n-gram, train RNN, verify artifacts.

In [ ]:
import csv, gzip, hashlib, json, re, sys
from pathlib import Path

ROOT = Path.cwd()
SPECIL_DIR = ROOT / 'SPECIL_dataset'
COMBINED = ROOT / 'specil_combined.csv'
TRAIN = ROOT / 'specil_train.csv'
TEST = ROOT / 'specil_test.csv'
SPACE_RE = re.compile(r'\s+')

In [ ]:
def clean(text):
    text = str(text).lower().replace('_', ' ')
    text = re.sub(r'\s*-\s*', '-', text)
    text = re.sub(r'([.,!?;:\"\'()])', r' \1 ', text)
    return SPACE_RE.sub(' ', text).strip()

def is_train(wrong, correct):
    key = f'{wrong}\n{correct}'.encode('utf-8')
    return int(hashlib.md5(key).hexdigest(), 16) % 100 < 80

def specil_rows():
    seen = set()
    for path in sorted(SPECIL_DIR.glob('*.csv')):
        with path.open(encoding='utf-8', newline='') as f:
            for row in csv.DictReader(f):
                wrong, correct = clean(row['kalimat_salah']), clean(row['kalimat_awal'])
                if not wrong or not correct or wrong == correct or (wrong, correct) in seen:
                    continue
                seen.add((wrong, correct))
                split = 'train' if is_train(wrong, correct) else 'test'
                yield {'wrong_text': wrong, 'correct_text': correct, 'error_type': row.get('tipe_kesalahan', ''), 'split': split}

def write_csv(path, rows):
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['wrong_text', 'correct_text', 'error_type', 'split'])
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
rows = list(specil_rows())
train_rows = [row for row in rows if row['split'] == 'train']
test_rows = [row for row in rows if row['split'] == 'test']
write_csv(COMBINED, rows)
write_csv(TRAIN, train_rows)
write_csv(TEST, test_rows)
len(rows), len(train_rows), len(test_rows)

In [ ]:
sys.path.insert(0, str(ROOT / 'n_gram'))
from ngram_spell_checker import MODEL_PATH as NGRAM_PATH, train as train_ngram

ngram = train_ngram(TRAIN)
ngram.save(NGRAM_PATH)
len(ngram.unigram), len(ngram.bigram), len(ngram.edits)

In [ ]:
sys.path.insert(0, str(ROOT / 'RNN'))
from specil_rnn_spellchecker import SpecilRnnSpellChecker

rnn = SpecilRnnSpellChecker.train(train_chars=250_000, epochs=1)
rnn.save()
rnn.split_counts, len(rnn.typo), len(rnn.vocab)

In [ ]:
meta = json.loads((ROOT / 'RNN' / 'specil_rnn_spellchecker_meta.json').read_text(encoding='utf-8'))
assert 'exact' not in meta
with gzip.open(ROOT / 'n_gram' / 'ngram_spell_checker.json.gz', 'rt', encoding='utf-8') as f:
    assert set(json.load(f)) == {'unigram', 'bigram', 'edits'}
assert ngram.correct('ak mw makn') == 'aku mau makan'
assert rnn.correct('ak mw makn') == 'aku mau makan'
print('ok')